In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
warnings.filterwarnings('ignore')

scaler = MinMaxScaler()
ohe = OneHotEncoder(sparse_output=False,drop='first',handle_unknown='ignore')

In [39]:
df = pd.read_csv(r'D:\Internship\Vehicle_Price_Prediction_working\data\featured_data.csv')

In [13]:
df.head()

,make,model,year,price,cylinders,fuel,mileage,transmission,trim,body,doors,exterior_color,interior_color,drivetrain,mileage_bucket,turbo,vehicle_age,annual_mileage
0,Jeep,Wagoneer,2024,74600.0,6,Gasoline,10.0,automatic,Series II,SUV,4,White,Global Black,Four-wheel Drive,brand new,0,1,10.0
1,Jeep,Grand Cherokee,2024,50170.0,6,Gasoline,1.0,automatic,Laredo,SUV,4,Metallic,Global Black,Four-wheel Drive,brand new,0,1,1.0
2,GMC,Yukon XL,2024,96410.0,8,Gasoline,0.0,automatic,Denali,SUV,4,Summit White,Teak/Light Shale,Four-wheel Drive,brand new,0,1,0.0
3,Dodge,Durango,2023,46835.0,8,Gasoline,32.0,automatic,Pursuit,SUV,4,White Knuckle Clearcoat,Black,All-wheel Drive,brand new,0,2,16.0
4,RAM,3500,2024,81663.0,6,Diesel,10.0,automatic,Laramie,Pickup Truck,4,Silver,Black,Four-wheel Drive,brand new,0,1,10.0


In [14]:
df.shape

(955, 18)

In [15]:
X = df.drop(columns=['price'])
y = df['price']

In [16]:
categorical_cols = [c for c in X.columns if X[c].dtype == 'object']
numerical_cols = [c for c in X.columns if X[c].dtype in ['int64', 'float64']]

In [17]:
preprocessor = ColumnTransformer(transformers=[
    ('Scaler',scaler,numerical_cols),
    ('OneHot',ohe,categorical_cols)
])

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [19]:
regressors = {
    "LinearRegression":LinearRegression(),
    "Ridge":Ridge(),
    "Lasso":Lasso(),
    "RandomForestRegressor":RandomForestRegressor(random_state=42,n_estimators=1000),
    "XGRegressor":XGBRegressor()
}

In [20]:
for name,regressor in regressors.items():
    model = Pipeline(steps=[
        ("Preprocessing",preprocessor),
        ("Regressor",regressor)
    ])

    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mse)

    print(name)
    print(f"\nAccuracy: {r2}\nMSE: {mse}\nMAE: {mae}\nRMSE: {rmse}")
    print()

LinearRegression

Accuracy: 0.8731749709897058
MSE: 55486380.23162667
MAE: 4386.70413734181
RMSE: 7448.918057787095

Ridge

Accuracy: 0.8585949100509662
MSE: 61865206.33054701
MAE: 4895.474031780961
RMSE: 7865.443810144919

Lasso

Accuracy: 0.8821595866777029
MSE: 51555580.40299994
MAE: 4232.090663502005
RMSE: 7180.221473116268

RandomForestRegressor

Accuracy: 0.8533586103183837
MSE: 64156105.218851246
MAE: 5074.488150379895
RMSE: 8009.7506339992415

XGRegressor

Accuracy: 0.8838139123022886
MSE: 50831807.33273923
MAE: 4484.651598903795
RMSE: 7129.6428615141185



In [21]:
params_xgb = {
    'n_estimators': [300, 500, 800],
    'max_depth': [4, 6, 8, 10],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
    'gamma': [0, 1, 5],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [0, 0.1, 1]
}

In [22]:
model_rscv_xgb = Pipeline(steps=[
    ('Pre',preprocessor),
    ('grid',RandomizedSearchCV(XGBRegressor(),param_distributions=params_xgb,cv=3,verbose=2,scoring='r2',random_state=42))
])

model_rscv_xgb.fit(X_train,y_train)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
[CV] END colsample_bytree=1.0, gamma=1, learning_rate=0.05, max_depth=6, n_estimators=800, reg_alpha=0, reg_lambda=1, subsample=0.8; total time=   0.8s
[CV] END colsample_bytree=1.0, gamma=1, learning_rate=0.05, max_depth=6, n_estimators=800, reg_alpha=0, reg_lambda=1, subsample=0.8; total time=   0.8s
[CV] END colsample_bytree=1.0, gamma=1, learning_rate=0.05, max_depth=6, n_estimators=800, reg_alpha=0, reg_lambda=1, subsample=0.8; total time=   0.9s
[CV] END colsample_bytree=0.7, gamma=0, learning_rate=0.1, max_depth=8, n_estimators=500, reg_alpha=1, reg_lambda=0.1, subsample=1.0; total time=   0.6s
[CV] END colsample_bytree=0.7, gamma=0, learning_rate=0.1, max_depth=8, n_estimators=500, reg_alpha=1, reg_lambda=0.1, subsample=1.0; total time=   0.6s
[CV] END colsample_bytree=0.7, gamma=0, learning_rate=0.1, max_depth=8, n_estimators=500, reg_alpha=1, reg_lambda=0.1, subsample=1.0; total time=   0.6s
[CV] END colsample_bytre

,steps,"[('Pre', ...), ('grid', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('Scaler', ...), ('OneHot', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [23]:
y_pred = model_rscv_xgb.predict(X_test)

r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mse)

In [24]:
model_rscv_xgb.named_steps['grid'].best_estimator_
# Model is selected for predictions

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [25]:
print(f"\nAccuracy: {r2}\nMSE: {mse}\nMAE: {mae}\nRMSE: {rmse}")


Accuracy: 0.8851551840265381
MSE: 50244996.405382
MAE: 4555.847429237565
RMSE: 7088.370504240167


In [26]:
params_lasso = {
    'alpha': [0.001, 0.01, 0.1, 1, 10],
    'max_iter': [3000, 5000],
    'tol': [1e-4],
    'selection': ['cyclic']
}


In [27]:
model_rscv_lasso = Pipeline(steps=[
    ('Pre',preprocessor),
    ('grid',RandomizedSearchCV(estimator=Lasso(),param_distributions=params_lasso,verbose=2,random_state=42,scoring='r2'))
])

In [28]:
model_rscv_lasso.fit(X_train,y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV] END alpha=0.001, max_iter=3000, selection=cyclic, tol=0.0001; total time=   0.6s
[CV] END alpha=0.001, max_iter=3000, selection=cyclic, tol=0.0001; total time=   0.6s
[CV] END alpha=0.001, max_iter=3000, selection=cyclic, tol=0.0001; total time=   0.6s
[CV] END alpha=0.001, max_iter=3000, selection=cyclic, tol=0.0001; total time=   0.6s
[CV] END alpha=0.001, max_iter=3000, selection=cyclic, tol=0.0001; total time=   0.5s
[CV] END alpha=0.001, max_iter=5000, selection=cyclic, tol=0.0001; total time=   1.3s
[CV] END alpha=0.001, max_iter=5000, selection=cyclic, tol=0.0001; total time=   1.1s
[CV] END alpha=0.001, max_iter=5000, selection=cyclic, tol=0.0001; total time=   1.0s
[CV] END alpha=0.001, max_iter=5000, selection=cyclic, tol=0.0001; total time=   1.2s
[CV] END alpha=0.001, max_iter=5000, selection=cyclic, tol=0.0001; total time=   1.0s
[CV] END alpha=0.01, max_iter=3000, selection=cyclic, tol=0.0001; total time=  

,steps,"[('Pre', ...), ('grid', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('Scaler', ...), ('OneHot', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [29]:
y_pred = model_rscv_lasso.predict(X_test)

r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mse)

In [30]:
print(f"\nAccuracy: {r2}\nMSE: {mse}\nMAE: {mae}\nRMSE: {rmse}")


Accuracy: 0.8821595866777029
MSE: 51555580.40299994
MAE: 4232.090663502005
RMSE: 7180.221473116268


In [31]:
import joblib

In [32]:
joblib.dump(model_rscv_xgb,"D:\Internship\Vehicle_Price_Prediction_working\model.pkl")

['D:\\Internship\\Vehicle_Price_Prediction_working\\model.pkl']